# Notebook 05 — Paper Figures

Generates all figures for the ColdGuard research paper:
1. Arrhenius rate curves for all 8 vaccines
2. Example DPT power-outage scenario (temperature + potency distribution)
3. Validation calibration plot (reliability diagram)
4. Simulation study results (from notebook 04)
5. Decision boundary visualization
6. Segment attribution heatmap

Run notebook 04 first to generate the simulation data.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from core.arrhenius import arrhenius_k, integrate_degradation
from core.vaccine_params import VACCINE_DB
from core.utils import parse_csv_log, run_analysis
from core.decision import Decision

FIGURES_DIR = Path('../paper/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Matplotlib style
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print("Setup complete. Figures will be saved to:", FIGURES_DIR.resolve())

## Figure 1 — Arrhenius Rate Curves (all 8 vaccines)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

T_range_C = np.linspace(-5, 45, 300)
T_range_K = T_range_C + 273.15

colors = plt.cm.tab10(np.linspace(0, 1, 8))

for color, (vtype, vp) in zip(colors, VACCINE_DB.items()):
    k_vals = [arrhenius_k(T, vp.Ea_mean, vp.A) for T in T_range_K]
    ax.semilogy(T_range_C, k_vals, label=vtype, color=color, linewidth=1.8)

ax.axvline(8.0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.text(8.5, ax.get_ylim()[0] * 1.5, 'Cold chain\nlimit (8°C)', fontsize=8, color='gray')

ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Degradation rate k [hr⁻¹]')
ax.set_title('Arrhenius degradation rate curves — UIP vaccines')
ax.legend(loc='upper left', ncol=2, fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_arrhenius_curves.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig1_arrhenius_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig1")

## Figure 2 — Example DPT Power-Outage Scenario

In [ ]:
import pandas as pd

ts, temps = parse_csv_log('../data/raw/demo_dpt_power_outage.csv')

result = run_analysis('DPT', ts, temps, n_mc_samples=5000)
dec = result['decision_output']
post = result['posterior_summary']
samples = result['potency_samples'] * 100

timestamps_dt = pd.to_datetime(ts, unit='s')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: temperature timeline
ax = axes[0]
ax.plot(timestamps_dt, temps, color='#1565C0', linewidth=2, marker='o', markersize=3)
ax.fill_between(timestamps_dt, temps, 8.0,
                where=np.array(temps) > 8.0,
                alpha=0.3, color='#EF5350', label='Excursion > 8°C')
ax.axhline(8.0, color='#EF5350', linestyle='--', linewidth=1, alpha=0.7, label='Upper limit (8°C)')
ax.axhline(0.0, color='#42A5F5', linestyle='--', linewidth=1, alpha=0.7, label='Freeze threshold (0°C)')
ax.set_xlabel('Time')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Temperature log — DPT power outage')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

# Right: potency distribution
ax = axes[1]
ax.hist(samples, bins=60, density=True, color='#90CAF9', edgecolor='white', linewidth=0.3)
ax.axvline(dec.estimated_potency_pct, color='#1565C0', linewidth=2, label=f'Mean {dec.estimated_potency_pct:.1f}%')
ax.axvline(dec.ci_90[0], color='#1565C0', linewidth=1.5, linestyle='--', label=f'90% CI [{dec.ci_90[0]:.1f}, {dec.ci_90[1]:.1f}]%')
ax.axvline(dec.ci_90[1], color='#1565C0', linewidth=1.5, linestyle='--')
ax.axvline(80.0, color='#E53935', linewidth=2, linestyle=':', label='Min. threshold (80%)')
dec_color = {'USE': '#4CAF50', 'INVESTIGATE': '#FF9800', 'DISCARD': '#F44336'}[dec.decision.value]
ax.set_title(f'Potency distribution — Decision: {dec.decision.value}', color=dec_color)
ax.set_xlabel('Remaining Potency (%)')
ax.set_ylabel('Density')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_example_scenario.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig2_example_scenario.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Decision: {dec.decision.value} | Potency: {dec.estimated_potency_pct:.1f}% (90% CI: {dec.ci_90[0]:.1f}–{dec.ci_90[1]:.1f}%)")

## Figure 3 — Decision Space (Potency vs Uncertainty)

In [ ]:
# Generate a grid of (mean_potency, std_potency) pairs and show decision zones
means = np.linspace(0.60, 1.0, 80)
stds = np.linspace(0.001, 0.10, 80)

from scipy.stats import norm

Z = np.zeros((len(stds), len(means)))
for i, std in enumerate(stds):
    for j, mean in enumerate(means):
        prob_above_80 = 1 - norm.cdf(0.80, loc=mean, scale=std)
        if prob_above_80 > 0.90:
            Z[i, j] = 2  # USE
        elif prob_above_80 > 0.70:
            Z[i, j] = 1  # INVESTIGATE
        else:
            Z[i, j] = 0  # DISCARD

fig, ax = plt.subplots(figsize=(8, 6))
cmap = plt.cm.colors.ListedColormap(['#FFCDD2', '#FFF9C4', '#C8E6C9'])
import matplotlib.colors as mcolors
cmap = mcolors.ListedColormap(['#EF9A9A', '#FFE082', '#A5D6A7'])
ax.contourf(means * 100, stds * 100, Z, levels=[-0.5, 0.5, 1.5, 2.5], cmap=cmap)

patches = [
    mpatches.Patch(color='#EF9A9A', label='DISCARD'),
    mpatches.Patch(color='#FFE082', label='INVESTIGATE'),
    mpatches.Patch(color='#A5D6A7', label='USE'),
]
ax.legend(handles=patches, loc='upper left')
ax.set_xlabel('Mean estimated potency (%)')
ax.set_ylabel('Potency std deviation (%, proxy for uncertainty)')
ax.set_title('ColdGuard decision zones (DPT, 80% threshold)')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_decision_space.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig3_decision_space.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig3")

## Figure 4 — Segment Attribution for Example Scenario

In [ ]:
import pandas as pd

segs = result['segment_attribution']
seg_df = pd.DataFrame(segs)
seg_df['start_dt'] = pd.to_datetime(seg_df['start_ts'], unit='s')
seg_df['pct'] = seg_df['degradation_fraction'] * 100

# Keep only top segments by contribution
seg_df_sorted = seg_df[seg_df['pct'] > 0.5].sort_values('pct', ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(seg_df_sorted) * 0.5)))
bars = ax.barh(range(len(seg_df_sorted)), seg_df_sorted['pct'], color='#EF5350')
ax.set_yticks(range(len(seg_df_sorted)))
ax.set_yticklabels([
    f"{row['start_dt'].strftime('%H:%M')} ({row['mean_temp_C']:.1f}°C, {row['duration_hours']:.1f}h)"
    for _, row in seg_df_sorted.iterrows()
], fontsize=9)
ax.set_xlabel('Share of total degradation (%)')
ax.set_title('Degradation attribution by time segment')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_segment_attribution.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig4_segment_attribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig4")

## Summary

Figures saved to `paper/figures/`:
- `fig1_arrhenius_curves.{pdf,png}` — Arrhenius rate curves for 8 vaccines
- `fig2_example_scenario.{pdf,png}` — DPT power-outage temperature + potency distribution
- `fig3_decision_space.{pdf,png}` — Decision zone contour plot
- `fig4_segment_attribution.{pdf,png}` — Degradation attribution by segment
- `fig_simulation_results.{pdf,png}` — (from notebook 04)

Update `paper/main.tex` to include these with `\\includegraphics`.